# hist

> Converting dialogs into LLM chat history

In [ ]:
#| default_exp hist

## Imports

In [ ]:
#| export
import re, ast, base64, binascii, hashlib
from ast import literal_eval
from datetime import datetime
from zoneinfo import ZoneInfo
from itertools import chain
from io import BytesIO

from fastcore.utils import *
from fastcore.xtras import detect_mime
from fastcore.nbio import item2xml, IMG_MIMES
from fastcore.xml import to_xml, Media, MediaUnavailable, Instructions, Prompt, Variable, Variables, System_reminder
from aidialog.msg_parts import MediaUrl, fmt2hist, hist2fmt, data_url, Msg, mk_tr_details, PartType, Part, Text, Thinking, ToolUse, ToolResult, InputImage, mk_content
from aidialog.dialog import *

In [ ]:
from aidialog.msg_parts import hist2fmt, mk_tool_res_msg
from fastcore.test import *
import random, json, tempfile
from IPython.display import Markdown

In [ ]:
def tool():
    "dummmy tool"
    pass

In [ ]:
random.seed(6)

### Overview

| Function | Purpose | Input → Output |
|----------|---------|----------------|
| `ai_fmt` | Format an AI reply for chat history | `str` → `str` |
| `Message.hist_xml` | Convert message to XML | `Message` → `str` |
| `Message.to_media` | Gather media from message | `Message` → `list[str\|bytes]` |
| `Message.to_parts` | Convert message to history parts | `Message` → `list[str\|bytes]` |
| `dlg2hist` | **Main entry point** | `list[Message], dict` → `hist` |

## Test helpers

`mk` is a convenience function to create conformant dummy data for testing. For `msg_type=='code'` cells, output text is a serialized structure following nbformat v4. For creating tests, we have a helper function `jwrap` that wraps output text in this jupyter v4 format.

In [ ]:
dname = 'backend'
dlg = Dialog(name=dname)

In [ ]:
#| export
def jwrap(txts:list=['']):
    "Wrap plain `txt` in a minimal Jupyter-style cell output"
    return [dict(output_type='stream', name='stdout', text=txt) for txt in listify(txts)]

In [ ]:
def mk(msg_type: str, content=None, output='', id=None, attachments=None):
    "Return a `Message`, ensuring `output` is always valid JSON. Uses timestamp-based id if not provided."
    if msg_type==scode: output = jwrap(output)
    if msg_type==sprompt: output = prompt_output(output)
    res = Message(msg_type=msg_type, output=output, id=id, attachments=attachments)
    res.content = content or f'{msg_type}>{res.id}'
    res.dlg = dlg # to avoid weakref issue
    return res

In [ ]:
jwrap("test output")

[{'output_type': 'stream', 'name': 'stdout', 'text': 'test output'}]

Code cells automatically `jwrap` the output value:

In [ ]:
mk('code', 'print("hello")', 'hello\n')

29f88512:c:print("hello") ⇒ out(64)

In [ ]:
mk('prompt', 'What is 2+2?', 'The answer is 4')

004af0bf:p:What is 2+2?
> The answer is 4

In [ ]:
#| export
def to_local_time(time_str, tz='UTC'):
    "Convert ISO time string to local timezone"
    if not time_str: return ''
    try: dt = datetime.fromisoformat(time_str)
    except ValueError: return ''
    return dt.astimezone(ZoneInfo(tz)).isoformat()

@patch
def local_time(self:Message):
    "Localized `time_run` for hosts that stamp one; '' when absent"
    if not (t := getattr(self, 'time_run', '')): return ''
    tz = getattr(self.dlg, 'timezone', 'UTC') if self.dlg else 'UTC'
    return to_local_time(t, tz)

In [ ]:
test_eq(to_local_time(''), '')
test_eq(to_local_time(None), '')
result = to_local_time('2026-01-27T12:00:00+00:00', 'UTC')
assert '2026-01-27' in result

m = Message('test', msg_type=snote)
test_eq(m.local_time(), '')

m.time_run = '2026-01-27T12:00:00+00:00'
assert m.local_time()

## Media

Media in the LLM context is represented by an XML metadata tag followed by raw bytes.

For successfully loaded media, the context contains:

- a `<media>` tag with `id`, `type`, and `mime` attributes
- the raw media bytes immediately after the tag

If media cannot be loaded or decoded, the context contains a `<media-unavailable>` tag instead of silently dropping it. This lets the AI tell the user that the media was referenced but was not available.

`id=` is:

- UUID for attachments referenced via `attachment:uuid` in markdown
- variable name for media in variables
- content hash for code output images

`type=` is one of:

- `output` — media from code cell output
- `variable` — media data in a variable
- `content` — media embedded in markdown message content

### Media context tags

Media are included via a `<media>` tag followed immediately by the raw bytes. The tag gives the AI a stable id, media type, and MIME type.

For unavailable media, we add a `<media-unavailable>` tag instead. This is used when media was referenced by the user but could not be loaded, decoded, resized, or safely resolved. The tags instructs the AI to tell the user that it could not see the referenced file.


Model info has the following fields `supports_vision`, `supports_video_input`, `supports_pdf_input`, `supports_audio_input` to indicate specific media type support

In [ ]:
#| export
def _mime_kind(mime):
    if mime == 'application/pdf': return 'pdf'
    return mime.split('/', 1)[0] if mime and mime.split('/', 1)[0] in {'image','audio','video'} else None

def _mime_supported(mime, aim_info):
    if   _mime_kind(mime) == 'image' and aim_info.get('supports_vision', False):      return True
    elif _mime_kind(mime) == 'video' and aim_info.get('supports_video_input', False): return True
    elif _mime_kind(mime) == 'audio' and aim_info.get('supports_audio_input', False): return True
    elif _mime_kind(mime) == 'pdf'   and aim_info.get('supports_pdf_input', False):   return True
    return False

In [ ]:
#| export
class _MissError(Exception): pass

UNSUPPORTED_MSG = "An unsupported media type was included in the context: {mime}. Please instruct the user to switch to a model that supports this media type."
Message.UNSUPPORTED_MSG = UNSUPPORTED_MSG

In [ ]:
#| export
def _media_xml(media_type, media_id, mime=None):
    "Create a media XML tag for LLM context."
    return to_xml(Media(id=media_id, type=media_type, mime=mime))

def _media_unavailable(media_type, media_id, e, mime=None):
    "Create a media-unavailable XML tag for LLM context."
    msg = "The user included this media, but it was not available to the AI. Tell the user that you cannot see it."
    return to_xml(MediaUnavailable(msg, id=str(media_id), type=media_type, reason=str(e), mime=mime))

def media_item(media_type, media_id, data, aim_info, mime=None, max_im_sz=None, prep=None, unavail_msg=UNSUPPORTED_MSG):
    "Load/prepare/package media, returning [xml_tag, data] or [media-unavailable]."
    try:
        # `data` can be bytes, or a function that loads bytes lazily.
        data = data() if callable(data) else data
        if data is None: raise _MissError("media data unavailable")
        if not isinstance(data, (bytes, MediaUrl)): raise _MissError(f"expected bytes or MediaUrl, got {type(data).__name__}")
        mime = mime or (detect_mime(data) if isinstance(data, bytes) else data.mime)
        if not mime: raise _MissError("unsupported or unknown media type")
        if not _mime_supported(mime, aim_info): raise _MissError(unavail_msg.format(mime=mime))
        # `media_id` can be a string, or a function that derives an id.
        media_id = media_id(data) if callable(media_id) else media_id
        if prep and isinstance(data, bytes): data = prep(data, mime, max_im_sz)
        return [_media_xml(media_type, media_id, mime), data]
    except (_MissError, OSError, binascii.Error) as e:
        media_id = getattr(media_id, '__name__', 'unknown') if callable(media_id) else media_id
        return [_media_unavailable(media_type, media_id, e, mime)]

In [ ]:
detect_mime('https://image.jpg')

A helper function and small dummy img for testing purposes.

In [ ]:
def _mk_img(imgb):
    "Image bytes from a base64 str"
    return base64.b64decode(imgb)

In [ ]:
_imgb64 = b'iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAAAAAA6fptVAAAACklEQVR4nGP4DwABAQEAsTj2FAAAAABJRU5ErkJggg=='

Valid media should produce a `<media>` tag plus bytes

In [ ]:
png = _mk_img(_imgb64)
res = media_item('content', 'img', png, dict(supports_vision=True))
test_eq(res[0], '<media id="img" type="content" mime="image/png"></media>')
test(res[1], bytes, isinstance)

Expected media-loading failures should produce `<media-unavailable>`

In [ ]:
for data in (None, 'not bytes', b'not media'): test(media_item('content', 'x', data, {})[0], '<media-unavailable', operator.contains)

def bad_b64_loader(): raise binascii.Error('bad base64')
test(media_item('output', 'bad', bad_b64_loader, {})[0], '<media-unavailable', operator.contains)

In [ ]:
def bad_loader(): raise RuntimeError('boom')
with expect_fail(Exception, 'boom'): media_item('content', 'x', bad_loader, {})

In [ ]:
test(media_item('content', 'img', png, dict(supports_vision=False))[0], "An unsupported media type was", operator.contains)
test(media_item('content', 'img', png, dict(supports_vision=False))[0], 'mime="image/png"', operator.contains)

These are attachments found in a message such as the ones uploaded from the clipboard or the screenshot of the last output.

In [ ]:
#| export
im_max = 768**2   # image-area budget for LLM context: OpenAI high detail = exactly 4 tiles; Anthropic ~786 toks
IMG_TOKS = 765    # assumed tokens per context image at `im_max`: OpenAI 85 base + 4*170 per tile

def resize_img(data:bytes, max_im_sz=im_max):
    "Resize `data` so that its area is <= `max_im_sz` total pixels; passthrough without pillow"
    try: from PIL import Image as PImg
    except ImportError: return data
    img = PImg.open(BytesIO(data))
    w,h = img.size
    if w*h <= max_im_sz: return data
    scale = (max_im_sz/(w*h))**0.5
    rw,rh = int(w*scale),int(h*scale)
    img_r = img.resize((rw, rh), PImg.Resampling.LANCZOS)
    img_b = BytesIO()
    img_r.save(img_b, format=img.format)
    return img_b.getvalue()

@patch
def prep_img(self:Message, data, mime, max_im_sz=None):
    "Resize raster images before they enter LLM context, so they don't consume too many tokens"
    return resize_img(data, max_im_sz or im_max) if mime in IMG_MIMES else data

In [ ]:
#| export
def _media_atts(msg, aim_info, max_im_sz=None):
    "Build media context from a message's attachments that are referenced in the content."
    attids = set()
    attids.update(re.findall(r'!\[[^\]]*\]\(attachment:([0-9a-f-]{36})\)', msg.content))
    attids.update(re.findall(r'<!-- last_output:([0-9a-f-]{36}) -->', msg.content))
    atts = L(msg.attachments).filter(lambda o: o in sorted(attids))
    return list(chain.from_iterable(media_item('content', o.id, o.data, aim_info=aim_info, max_im_sz=max_im_sz,
        prep=msg.prep_img, unavail_msg=msg.UNSUPPORTED_MSG) for o in atts))

In [ ]:
def parse_att_id(s):
    "Extract id attribute from XML tag string"
    return re.search(r' id="([^"]*)"', s).group(1)

In [ ]:
_mk_img(_imgb64)[:40]

b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00\x01\x08\x00\x00\x00\x00:~\x9bU\x00\x00\x00\nIDA'

Only valid image attachments should be included:

In [ ]:
att1,att2,att3 = Attachment(png, 'image/png'), Attachment(b'not an image', 'text/plain'), Attachment(_mk_img(_imgb64), 'image/png')
m = Message(f'![](attachment:{att1.id}) and ![](attachment:{att3.id})', attachments=[att1, att2, att3])
res = _media_atts(m, dict(supports_vision=True))

In [ ]:
test_eq(len(m.attachments), 3)
test_eq(len(res), 4)
test_eq(L(res[::2]).map(parse_att_id), [att1.id, att3.id])

Only the attachments referenced in the message content should be included:

In [ ]:
m = Message(f'Only ![](attachment:{att3.id})', attachments=[att1,att3])
res = _media_atts(m, dict(supports_vision=True))

test_eq(len(m.attachments), 2)
test_eq(L(res[::2]).map(parse_att_id), [att3.id])

In [ ]:
bad = Attachment(b'not an image', 'image/png')
res = _media_atts(Message(f'![](attachment:{bad.id})', attachments=[bad]), dict(supports_vision=True))
test(res[0], '<media-unavailable', operator.contains)

These are images from the message outputs, such as a code output displaying a plot

In [ ]:
#| export
def _img_id(im_bytes):
    "Generate a deterministic 8 char id from `im_bytes`"
    h = hashlib.md5(im_bytes)
    return base64.b32encode(h.digest()).decode('utf-8')[:8]

In [ ]:
test_eq(_img_id(_mk_img(_imgb64)), _img_id(_mk_img(_imgb64)))
_img_id(_mk_img(_imgb64))

'4P52II3F'

In [ ]:
#| export
def _img_item(pair, m, aim_info, max_im_sz):
    mime,data = pair
    return media_item('output', _img_id, partial(base64.b64decode, data), aim_info, mime, max_im_sz, prep=m.prep_img, unavail_msg=m.UNSUPPORTED_MSG)

def _out_img(data):
    "Preferred `(mime, b64)` image entry of mimebundle `data`, or None"
    return first((m, data[m]) for m in IMG_MIMES if m in data)

def _img_output(m, aim_info, max_im_sz):
    "Extract image outputs from `m` as encoded images and their xml tags, at most one image per output"
    pairs = L(m.output or []).attrgot('data', {}).map(_out_img).filter()
    return pairs.flatmap(_img_item, m=m, aim_info=aim_info, max_im_sz=max_im_sz)

In [ ]:
img_out = mk_code_output({'image/png': base64.b64encode(png).decode()})
m = Message(msg_type='code', output=img_out)

In [ ]:
res = _img_output(m, dict(supports_vision=True), None)
test_eq(len(res), 2)
test_eq(res[0], _media_xml('output', _img_id(png), 'image/png'))

In [ ]:
m = Message(msg_type='code', output=mk_code_output({'image/png': 'not-base64'}))
res = _img_output(m, dict(supports_vision=True), None)
test(res[0], '<media-unavailable', operator.contains)

These two are the public face of the outputs-media machinery, shared by every host that returns code results to a model (solveit's tool results, clikernel's MCP frontend). `output_parts` runs the full pipeline — capability gate, resize, id tag, unavailable fallback — and hands back finished `Part` objects. `merge_media` is the composition policy: media first with each `<media>` tag adjacent to its image, rendered text last; when no image survived the gate, the unavailable notes fold into the text and the result stays a plain string. The `str | list[Part]` return is deliberate: "plain text result" and "multipart result" are semantically different, and callers already branch on exactly that.

In [ ]:
#| export
def output_parts(m, aim_info=None, max_im_sz=None):
    "Media `Part`s for a code message's outputs: gated, resized, id-tagged, with unavailable fallbacks; images enabled if `aim_info` is None"
    if aim_info is None: aim_info = dict(supports_vision=True)
    return [mk_content(o) for o in _img_output(m, aim_info, max_im_sz)]

def merge_media(text, parts):
    "Compose rendered `text` with media `parts`: media first (tag adjacency kept), text last; with no images, notes fold into the text"
    has_img = any(isinstance(p, InputImage) for p in parts)
    if not has_img: return '\n\n'.join(filter(None, [text, *(p.text for p in parts)]))
    return [*parts, *([Text(text)] if text else [])]

In [ ]:
parts = output_parts(m := Message(msg_type='code', output=mk_code_output({'image/png': base64.b64encode(png).decode()})))
test_eq([type(p).__name__ for p in parts], ['Text', 'InputImage'])
test(parts[0].text, '<media', operator.contains)
test_eq(data_url(parts[1].text)[0], 'image/png')

In [ ]:
# One image per output: with several raster mimes in one bundle, `IMG_MIMES` order picks the winner
test_eq(_out_img({'image/png': 'AA', 'image/jpeg': 'BB', 'image/webp': 'CC'}), ('image/jpeg', 'BB'))
test_eq(_out_img({'text/plain': 'x'}), None)
both = mk_code_output({'image/png': base64.b64encode(png).decode(), 'image/jpeg': base64.b64encode(png).decode()})
parts = output_parts(Message(msg_type='code', output=both))
test_eq(len(parts), 2)


With an image present, `merge_media` returns a list with the text last; with vision off, the media notes fold into a plain string instead.

In [ ]:
res = merge_media('42', parts)
test_eq([type(p).__name__ for p in res], ['Text', 'InputImage', 'Text'])
test_eq(res[-1].text, '42')
folded = merge_media('42', output_parts(Message(msg_type='code', output=both), aim_info={}))
test_eq(type(folded), str)
test(folded, 'An unsupported media type', operator.contains)
test_eq(merge_media('42', []), '42')
res[-1]

**Text** (`text`)

42

::: details

- raw: `None`
- citations: `None`

:::

### Static media refs

Messages can pull media straight from markdown image links: `![alt](ref#ai)` marks its target for AI context, and a link without the `#ai` anchor stays invisible to the model. Refs may be:

- web URLs (`http://...`, `https://...`), passed through lazily as `MediaUrl`
- `data:` URLs, decoded inline
- file paths, resolved by `Dialog.media_path`: relative paths resolve against the dialog's directory

`media_path` is the host hook: solveit patches it to map its `/static/` prefix and enforce its prod data-dir sandbox, and any host with its own roots or safety rules patches this one function.

In [ ]:
#| export
@patch
def media_path(self:Dialog, ref):
    "Resolve a non-URL `#ai` media ref to a local path; hosts patch this to add their own roots and safety rules"
    p = Path(ref)
    if not p.is_absolute() and (pth := getattr(self, 'path_', None)): p = Path(pth).parent/p
    return p.resolve()

def _mk_media_tag(ref, msg, aim_info, max_im_sz=None):
    "Media tag and bytes for a path, URL, or base64 `ref`; raster images resize via `Message.prep_img`"
    kw = dict(prep=msg.prep_img, unavail_msg=msg.UNSUPPORTED_MSG)
    if ref.startswith('data:'):
        meta,data = ref.split(',', 1)
        mime = meta.removeprefix('data:').split(';')[0]
        return media_item('content', data[:10], lambda: base64.b64decode(data), aim_info, mime, max_im_sz, **kw)
    data = lambda: MediaUrl(ref) if ref.startswith(('http://', 'https://')) else (msg.dlg.media_path(ref) if msg.dlg else Path(ref).resolve()).read_bytes()
    return media_item('content', ref, data, aim_info, max_im_sz=max_im_sz, **kw)

_static_pat = re.compile(r'!\[[^\]]*\]\(([^)#]+)#ai\)')
def _media_static(msg, aim_info, max_im_sz=None):
    "Media items for `#ai`-tagged markdown links in `msg.content`"
    if '#ai' not in msg.content: return []
    return list(chain.from_iterable(_mk_media_tag(ref, msg, aim_info, max_im_sz) for ref in _static_pat.findall(msg.content)))

The default `media_path` resolves relative refs against the dialog's own directory (`path_`, stamped by `create_dlg`/`read_ipynb`); absolute paths pass through untouched:

In [ ]:
tdir = Path(tempfile.mkdtemp()).resolve()
(tdir/'sq.png').write_bytes(png)
stdlg = Dialog(name='statics')
stdlg.path_ = str(tdir/'statics.ipynb')
test_eq(stdlg.media_path('sq.png'), tdir/'sq.png')
test_eq(stdlg.media_path('/abs/x.png'), Path('/abs/x.png'))


`_media_static` collects the `#ai`-tagged refs in order: a local file becomes a `<media>` tag plus (resized) bytes, a web URL a tag plus a lazy `MediaUrl`, a `data:` URL a tag plus decoded bytes, and an unreadable ref a `<media-unavailable>` tag. Untagged links are skipped:

In [ ]:
m = Message('some text', dlg=stdlg)
test_eq(_media_static(m, {}), [])
m = Message('a ![](sq.png#ai), a skipped ![](sq.png), and ![](https://ex.org/logo.png#ai)', dlg=stdlg)
res = _media_static(m, dict(supports_vision=True))
test_eq(len(res), 4)
test_eq(res[0], '<media id="sq.png" type="content" mime="image/png"></media>')
test(res[1], bytes, isinstance)
test_eq(res[2], '<media id="https://ex.org/logo.png" type="content" mime="image/png"></media>')
test(res[3], MediaUrl, isinstance)

In [ ]:
m = Message('A tiny square ![Square](data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAoAAAAKCAIAAAACUFjqAAAAE0lEQVR4nGP8z4APMOGVZRip0gBBLAETee26JgAAAABJRU5ErkJggg==#ai) and a bad ref ![](nope.png#ai)', dlg=stdlg)
res = _media_static(m, dict(supports_vision=True))
test_eq(len(res), 3)
test_eq(res[0], '<media id="iVBORw0KGg" type="content" mime="image/png"></media>')
test(res[1], bytes, isinstance)
test(res[2], '<media-unavailable', operator.contains)

### The `mk_msgs` contract

The parts lists built in this module exist to feed `mk_msgs` (from `aidialog.msg_parts`), so its reading of nested lists is a load-bearing assumption; these cells pin it where the assumption lives. A flat list is one exchange, alternating user and assistant:


In [ ]:
from aidialog.msg_parts import mk_msgs, mk_msg
from fastcore import imghdr


In [ ]:
mk_msgs(['hello', "here's a response"])

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='hello', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="here's a response", citations=None)])]

A list element becomes a content container, so one user turn can carry several parts:

In [ ]:
mk_msgs([['hello1', 'hello2', 'hello3'], "here's a response"])

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='hello1', citations=None), Text(raw=None, cache_control=None, text='hello2', citations=None), Text(raw=None, cache_control=None, text='hello3', citations=None)]),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text="here's a response", citations=None)])]

Within a container, any element that `imghdr.what` recognizes as image bytes becomes an image part rather than text — this is how the raw bytes following each `<media>` tag reach the model:

In [ ]:
test_eq(imghdr.what(None, b'not an image'), None)
test_eq(imghdr.what(None, png), 'png')
mk_msg(['hello', png, 'hello3'])

**Msg**

- role: `user`

<contents>

**Text** (`text`)

hello

::: details

- raw: `None`
- citations: `None`

:::

**InputImage** (`input_image`)

data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAAAAAA6fptVAAAACklEQVR4nGP4DwABAQEAsTj2FAAAAABJRU5ErkJggg==

::: details

- raw: `None`
- mime: `image/png`

:::

**Text** (`text`)

hello3

::: details

- raw: `None`
- citations: `None`

:::

</contents>

## Sigil refs and variables

Dialog content can reference things by sigil: $\`expr\` interpolates a kernel expression's value, &\`tool\` activates a tool, and !\`cmd\` (in hosts with a shell) references a shell command's output. Parsing is shared here; evaluation is host-side, since only the host knows its kernel, toolset, and shell. Two grammars cover the three sigils: name refs (`&`, via `get_refs`, which also accepts a `[a, b]` list form) and payload refs (`$` and `!`, via `sigil_pat`, whose backtick body is an arbitrary expression or command).

In [ ]:
#| export
def get_refs(cts, sigil='&'):
    'Return sorted unique names referenced via sigil`name` or sigil`[name, name]` pattern in msgs'
    ms = re.findall(rf'{re.escape(sigil)}`([\w.]+|\[[\w.,\s]+\])`', cts)
    if not ms: return None
    return sorted(set(t.strip() for m in ms for t in re.split(r'[,\[\]]', m) if t.strip()))

In [ ]:
test_eq(get_refs('use &`bash` and &`[python, bash]`'), ['bash', 'python'])
test_eq(get_refs('dotted names too: &`np.load`'), ['np.load'])
test_is(get_refs('no refs here, and no $`vars` either'), None)

Variable parsing and construction are decoupled so each basic operation works independently. `get_exprs` scans prompt messages only (`expr_mtypes`): a $\`...\` in a note or a code output is discussion, not a request to interpolate. The `sigil` param applies the same scan to other ref forms: `get_exprs(msgs, sigil='!')` collects shell refs, with identical message-type rules.

In [ ]:
#| export
def sigil_pat(sigil):
    "Regex for sigil`payload` refs, e.g. $`expr` or !`cmd`"
    return re.compile(rf'{re.escape(sigil)}`([^`]+)`')

expr_pat = sigil_pat('$')
expr_mtypes = ('prompt',)
def get_exprs(msgs, sigil='$'):
    "Sorted unique refs of `sigil` form found in prompt messages of `msgs`"
    cts = ' '.join((m.content or '') for m in msgs if m.msg_type in expr_mtypes)
    return sorted(set(sigil_pat(sigil).findall(cts)))


In [ ]:
test_eq(expr_pat.findall('$`myvar`'), ['myvar'])
test_eq(expr_pat.findall('$`now()`'), ['now()'])
test_eq(expr_pat.findall('$`add(1, 2)`'), ['add(1, 2)'])
test_eq(expr_pat.findall('$`[obj.attr, d["key"], fn(*args,**kws)]`'), ['[obj.attr, d["key"], fn(*args,**kws)]'])
test_eq(expr_pat.findall('Check $`x` and $`f(y)`'), ['x', 'f(y)'])

Adversarial cases: $\`...\` in notes and code cells is ignored — only prompts request interpolation:

In [ ]:
raw = [mk('prompt', '>hello $`x`'), mk('note', '$`not_variable1`'), mk('code', output="aaa", content="'$`not_variable2`'"),
    mk('prompt', 'hi <hi $`y`'), mk('prompt', '$`x`, $`y`')]
test_eq(get_exprs(raw), ['x', 'y'])
raw += [mk('prompt', 'run !`ls -la` and !`git status`'), mk('note', '!`not_a_cmd`')]
test_eq(get_exprs(raw, sigil='!'), ['git status', 'ls -la'])
test_eq(get_exprs(raw), ['x', 'y'])  # the default is unaffected by ! refs


Parsing gives the names; the other half is rendering their *values* into context. The values come from the host's kernel: it evaluates the referenced expressions and passes the resulting dict-style namespace to the builders below. Text values ride in a `<variables>` XML element (built from fastcore's `Variables`/`Variable` components), while byte values such as images become interleaved media parts. `vars_tag` builds both:

In [ ]:
#| export
def vars_tag(aim_info, ns=None):
    "<Variables> element and byte variables for this message"
    if not ns: return {}
    lead = f"The 'variables' section provides values of: {', '.join(v if v.startswith(('$`','!`')) else f'$`{v}`' for v in ns)}"
    vs,vimgs = [],[]
    for k,v in ns.items():
        if mime := detect_mime(v): vimgs.extend(media_item('variable', k, v, aim_info, mime))
        else: vs.append(Variable(v, name=k, type=type(v).__name__))
    vs = Variables(lead, *vs) if vs else None
    return dict(vars_tag=vs, vars_bytes=vimgs)

In [ ]:
imns = dict(img=_mk_img(_imgb64), x=4, y=3, img2=_mk_img(_imgb64))
vt = vars_tag(dict(supports_vision=True), imns)
test_eq(to_xml(vt['vars_tag']), '''<variables>The 'variables' section provides values of: $`img`, $`x`, $`y`, $`img2`<variable name="x" type="int">4</variable><variable name="y" type="int">3</variable></variables>''')

Byte vars become interleaved `<media>` tags and bytes (image, audio, video), ready for a `mk_msgs` content container:

In [ ]:
test_eq(len(vt['vars_bytes']), 4)
test_eq(vt['vars_bytes'][0], '<media id="img" type="variable" mime="image/png"></media>')
test_eq(vt['vars_bytes'][2], '<media id="img2" type="variable" mime="image/png"></media>')

A `!`-sigil key (a shell ref, evaluated by hosts with a shell) passes through the lead verbatim and renders as an ordinary variable whose name is the full ref — one section, one stable position, whatever the sigil:

In [ ]:
vt = vars_tag({}, {'x': 4, '!`ls -la`': 'total 0'})
test_eq(to_xml(vt['vars_tag']), '''<variables>The 'variables' section provides values of: $`x`, !`ls -la`<variable name="x" type="int">4</variable><variable name="!`ls -la`" type="str">total 0</variable></variables>''')

`vars_hist` packages the whole namespace as one synthetic exchange — every value in a single user turn, answered by a brief acknowledgement. Hosts place that turn above everything else in history, so variable values appear exactly once, in a stable position that caches well:

In [ ]:
#| export
def vars_hist(aim_info, ns=None):
    "Single history turn holding all variable values, to sit above everything else"
    vt = vars_tag(aim_info, ns)
    if not vt: return []
    parts = []
    if vt.get('vars_tag'): parts.append(to_xml(vt['vars_tag'], do_escape=False))
    if vt.get('vars_bytes'): parts.extend(vt['vars_bytes'])
    if not parts: return []
    return [parts, 'Understood, I will use these variable values.']

In [ ]:
test_eq(vars_hist({}, {}), [])
test_eq(vars_hist({}, None), [])
vh = vars_hist({}, {'x': 4, 'y': 3})
test_eq(len(vh), 2)
test_eq(vh[0][0], """<variables>The 'variables' section provides values of: $`x`, $`y`<variable name="x" type="int">4</variable><variable name="y" type="int">3</variable></variables>""")
vh = vars_hist(dict(supports_vision=True), {'img': _mk_img(_imgb64), 'x': 4})
test_eq(len(vh[0]), 3)
test_eq(vh[0][1], '<media id="img" type="variable" mime="image/png"></media>')

Hosts fill the namespace by evaluating the parsed refs kernel-side (ipyfuncs' `eval_exprs`, via each client's sugar). That call returns a value for *every* requested expression — a failing one comes back as an `<error type="..." .../>` string, never as an absent key. `is_nameerr` is the marker test for the undefined-name case, so hosts can route those into a missing-vars warning instead of rendering the error string as a variable's "value". Other error types stay visible as values deliberately: a `$`-ref that raises is information, a name that doesn't exist is a typo.

In [ ]:
#| export
def is_nameerr(v):
    "Does an `eval_exprs` result mark an undefined name?"
    return isinstance(v, str) and v.startswith('<error type="NameError"')

## Message XML Helpers

These methods convert `Message` objects into XML for LLM context, using the shared `item2xml` grammar (per-type tags, content bare inside, an `<out>` section when output is present). Prompts render as their bare content via the `prompt_txt` host hook: in history, the reply is the next turn, not part of the message.

### The prompt envelope

A prompt does not render as bare content: every prompt is wrapped in an envelope — an `<instructions>` element telling the model how to treat what follows, and a `<prompt>` element carrying the text with its message id (so tools can address the message) and its local run time. The envelope delimits the actual request from the context packed into the same user turn, since notes, code, and media all precede the prompt in a history turn's parts list. `warning_tag` wraps the rare context warnings (undefined variables, budget trims) in a `<system-reminder>`; hosts insert it as a sibling part of the live prompt.

In [ ]:
#| export
def task_tags(task: str, sid=None, time=None, tz='UTC'):
    "Envelope elements for a prompt: `<instructions>` plus `<prompt>` carrying `sid` and local run time"
    kw = {'sid':sid} if sid else {}
    if time: kw['time'] = to_local_time(time, tz)
    return Instructions('Respond to the request in the `prompt` below.'), Prompt(task, **kw)

def warning_tag(warning: str): return System_reminder('**NB**: ' + warning) if warning else None

In [ ]:
test_eq(to_xml(task_tags("do it.", "testsid")), '<instructions>Respond to the request in the `prompt` below.</instructions><prompt sid="testsid">do it.</prompt>')
test_eq(to_xml(warning_tag('The following vars are undefined: `foo`')), '<system-reminder>**NB**: The following vars are undefined: `foo`</system-reminder>')
test_is(warning_tag(''), None)

`hist_xml` combines the msg input and output. A prompt renders through `prompt_txt` — the envelope above, identical for every prompt (`last` marks the live request but doesn't change the rendering); a host that wants different prompt rendering patches `prompt_txt` alone:


In [ ]:
#| export
@patch
def prompt_txt(self:Message, last=False):
    "History text for a prompt: the envelope (`last`, threaded by `dlg2hist`, is unused: the envelope is the same for every prompt)"
    tz = getattr(self.dlg, 'timezone', 'UTC') if self.dlg else 'UTC'
    instr,task = task_tags(self.content, self.id, getattr(self, 'time_run', None), tz)
    return to_xml(instr, do_escape=False) + '\n' + to_xml(task, do_escape=False)

@patch
def hist_xml(self:Message, last=False):
    "History XML for this message (the concise `Message.to_xml` is dlgskill's converter; this rendering adds time and serves `to_parts`)"
    if self.msg_type == sprompt: return self.prompt_txt(last)
    it = item2xml('markdown' if self.msg_type==snote else self.msg_type, self.content, self.ai_output,
                  id=self.id, time=self.local_time() or None, meta=self.meta)
    return to_xml(it, do_escape=False)

In [ ]:
# Combines content and output into full XML representation
m = mk('code', 'print("hi")', 'hi\n')
test_eq(m.hist_xml(), f'<code id="{m.id}">print("hi")<out>hi\n</out></code>')
n = mk('note', '# hey')
test_eq(n.hist_xml(), f'<markdown id="{n.id}"># hey</markdown>')  # notes render as markdown; no out, no extra tags
m.meta = dict(nbdev=dict(export='true', default_exp='core'))
test_eq(m.hist_xml(), f'<code id="{m.id}" export default_exp="core">print("hi")<out>hi\n</out></code>')  # meta directives as attrs

In [ ]:
# Prompts render enveloped: instructions plus a prompt tag carrying the message id
m = mk('prompt', 'question', id='_t1')
test_eq(m.hist_xml(), '<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="_t1">question</prompt>')
test_eq(m.prompt_txt(last=True), m.hist_xml())


In [ ]:
#| export
@patch
def media_extra(self:Message, aim_info, max_im_sz=None):
    "Extra media sources beyond attachments and outputs: `#ai`-tagged markdown links by default; hosts patch to add more"
    return _media_static(self, aim_info, max_im_sz)

@patch
def to_media(self:Message, aim_info, max_im_sz=None):
    "Aggregate media from attachments, `media_extra`, and code outputs"
    media_ctx = []
    if media_atts := _media_atts(self, aim_info, max_im_sz): media_ctx += media_atts
    if extra := self.media_extra(aim_info, max_im_sz): media_ctx += extra
    if self.msg_type == scode:
        if img_outputs := _img_output(self, aim_info, max_im_sz): media_ctx += img_outputs
    return media_ctx

In [ ]:
# No media for plain messages
test_eq(mk('note', 'hello').to_media({}), [])

In [ ]:
# Code output images
m = Message(msg_type='code', output=mk_code_output({'image/png': base64.b64encode(png).decode()}))
media = m.to_media(dict(supports_vision=True))
test_eq(len(media), 2)  # xml tag, bytes

For a given message, media is collected in a fixed order: attachments, then `#ai` statics, then code output images. Media rides with the message it appears in, so the model sees images in dialog order:

In [ ]:
m = Message('a static ![](sq.png#ai)', dlg=stdlg)
media = m.to_media(dict(supports_vision=True))
test_eq(len(media), 2)
test_eq(media[0], '<media id="sq.png" type="content" mime="image/png"></media>')

In [ ]:
#| export
@patch
def to_parts(self:Message, aim_info:dict, last=False):
    "Convert message to a list of history parts: media, then XML text."
    parts = []
    if media := self.to_media(aim_info): parts.extend(media)
    if mxml := self.hist_xml(last): parts.append(mxml)
    return parts

In [ ]:
att = Attachment(_mk_img(_imgb64), 'image/png')
m = mk('prompt', f'What is in ![](attachment:{att.id})?', attachments=[att])
res = m.to_parts(dict(supports_vision=True))
['...bytes...' if isinstance(o,bytes) else o for o in res]

['<media id="112a8d8f-8cf2-4ce3-ae10-2b7f7a1058ac" type="content" mime="image/png"></media>',
 '...bytes...',
 '<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="101a6bdd">What is in ![](attachment:112a8d8f-8cf2-4ce3-ae10-2b7f7a1058ac)?</prompt>']

In [ ]:
#| export
def dlg2hist(
    dlg, # A `Dialog`, or iterable of messages, ending with a prompt
    aim_info:dict, # Model capability dict for media handling
    plain:bool=False, # Render prompts as bare content, without the serving envelope?
):
    "Convert `dlg` to LLM history. The final prompt renders with `last=True`."
    msgs = dlg.messages if isinstance(dlg, Dialog) else list(dlg)
    msgs = [m for m in msgs if not (m.msg_type==sraw and m.meta.get('rec_kind'))]  # tagged raws are session bookkeeping, not conversation
    msgs = [m for m in msgs if not m.skipped]  # `skipped` messages are hidden from the AI (solveit semantics)
    if msgs[-1].msg_type != sprompt: raise ValueError("dlg2hist requires the messages to end with a prompt")
    res = []
    for is_first, m in loop_first(reversed(msgs)):
        if m.msg_type == sprompt: res += [m.ai_output, (m.to_media(aim_info) or [])+[m.content] if plain else m.to_parts(aim_info, last=is_first)]
        else: res[-1] = m.to_parts(aim_info) + res[-1]
    return res[::-1]

## Dialogs to canonical messages

`dlg2hist` produces alternating user parts and AI reply strings. For surgery we usually want the replies' tool calls back as structured messages, and `msg_parts.fmt2hist` inverts the reply format exactly. `dlg2chat` composes the two into the canonical form every provider conversion starts from, and rejects duplicate tool-call ids, which hand-authored dialogs can accidentally produce. Thinking blocks survive only in the stored reply string: explode/implode (`reply2dlg`/`dlg2reply`) and this transmission projection both drop them, a deliberate loss since nothing downstream keeps them.

`reply2chat` is that inversion as a standalone step: one stored reply string to sequenced chat messages. Prompts render wrapped in the solveit serving envelope (`task_tags`) by default; `plain=True` renders them as their bare content instead, which is what transcript reconstruction wants.

In [ ]:
#| export
def _explode(a, t):
    "One call/result pair per tool use in a batched assistant/tool pair"
    tus = [p for p in a.content if isinstance(p, ToolUse)]
    if len(tus)<2: return [a,t]
    pre,trs = [p for p in a.content if not isinstance(p, ToolUse)],{p.id:p for p in t.content}
    res = []
    for is_first,tu in loop_first(tus):
        res += [Msg(role='assistant', content=(pre if is_first else [])+[tu]), Msg(role='tool', content=[trs[tu.id]])]
    return res

def _seq_tools(msgs):
    "Explode batched tool calls into strict call/result alternation"
    out = []
    for m in msgs:
        if m.role=='tool' and out and out[-1].role=='assistant': out += _explode(out.pop(), m)
        else: out.append(m)
    return out

In [ ]:
#| export
def reply2chat(outp:str)->list[Msg]:
    "Chat messages for a stored reply: `fmt2hist`-parsed parts in strict call/result alternation"
    return _seq_tools(fmt2hist(outp))

In [ ]:
#| export
def dlg2chat(
    dlg, # A `Dialog`, ending with a prompt
    aim_info=None, # Model capability dict for media handling; images enabled if None
    plain=False, # Render prompts as bare content, without the serving envelope?
):
    "Canonical chat messages for `dlg`, with each reply's tool calls recovered as real parts"
    hist = dlg2hist(dlg, dict(supports_vision=True) if aim_info is None else aim_info, plain=plain)
    msgs = []
    for i,t in enumerate(hist):
        if i%2==0: msgs.append(Msg('user', [mk_content(o) for o in t]))
        else: msgs += reply2chat(t)
    ids = [p.id for m in msgs for p in m.content if isinstance(p, ToolUse)]
    if dups := {i for i in ids if ids.count(i)>1}: raise ValueError(f"duplicate tool call id(s): {', '.join(sorted(dups))}")
    return msgs

The conversion tests below need replies carrying tool-call details blocks, so first a fixture that builds one in the wire reply format.

In [ ]:
def _mk_dtl(func, args, result, tid='x1'):
    d = json.dumps(dict(id=tid, name=func, args=args, result=result))
    return f"```json {{.tool}}\n{d}\n```"

py_dtl = _mk_dtl('python', {'code':'1+1'}, '2')

In [ ]:
ddlg = Dialog(name='canon')
ddlg.mk_message('What is 1+1?', msg_type=sprompt, output=f"Check:\n\n{py_dtl}\n\nIt is 2.")
cm = dlg2chat(ddlg)
test_eq([m.role for m in cm], ['user', 'assistant', 'tool', 'assistant'])
test_eq(cm[1].content[1].name, 'python')


One reply often holds several sequential tool calls, each issued after reading the previous result. `fmt2hist` returns them batched into a single call message and a single result message, which reads as one parallel volley; `dlg2chat` explodes the batch back into strict call/result alternation, the shape live sessions record.

In [ ]:
py_dtl2 = _mk_dtl('python', {'code':'2+2'}, '4', 'x2')
sdlg = Dialog(name='seq')
sdlg.mk_message('Two steps.', msg_type=sprompt, output=f"{py_dtl}\n\n{py_dtl2}\n\nDone.")
sm = dlg2chat(sdlg)
test_eq([(m.role, len(m.content)) for m in sm], [('user',1), ('assistant',1), ('tool',1), ('assistant',1), ('tool',1), ('assistant',1)])
test_eq(sm[2].content[0].id, sm[1].content[0].id)
test_eq(sm[4].content[0].id, sm[3].content[0].id)

In [ ]:
bad = Dialog(name='dup')
bad.mk_message('a', msg_type=sprompt, output=py_dtl)
bad.mk_message('b', msg_type=sprompt, output=py_dtl)
with expect_fail(ValueError, 'duplicate tool call id'): dlg2chat(bad)

`chat2dlg` is the inverse projection: canonical messages back to a dialog, one prompt per user turn, with each turn's assistant and tool messages rendered into the reply in the format `fmt2hist` parses. The round-trip is canonical rather than byte-exact - `hist2fmt` re-renders details blocks in its own style - so what `dlg2chat` recovers from the result is equal to what it recovered from the original. User image parts become attachments referenced from the question text. The prompt envelope is unwrapped: the reconstructed message carries the bare request, and the envelope's `sid` restores the original message id, so a re-projection wraps it identically.

In [ ]:
#| export
_env_pat = re.compile(r'<instructions>.*?</instructions>\n<prompt(?: sid="(?P<sid>[^"]*)")?(?: time="[^"]*")?>(?P<task>.*)</prompt>$', re.S)
def chat2dlg(
    msgs, # Canonical messages, e.g. from `dlg2chat`
    name, # Dialog name
    cls=Dialog, # Dialog class to create
    mx=2000, # Maximum characters per rendered tool input/output string; None disables truncation (see `hist2fmt`)
):
    "A dialog for `msgs`: one prompt per user turn, replies rendered in the format `fmt2hist` parses"
    dlg,turns = cls(name=name),[]
    for m in msgs:
        if m.role=='user': turns.append((m,[]))
        elif turns: turns[-1][1].append(m)
        else: turns.append((Msg('user', []), [m]))
    for i,(u,replies) in enumerate(turns):
        segs,atts = [],[]
        for p in u.content:
            if isinstance(p, Text): segs.append(p.text)
            elif isinstance(p, InputImage):
                mime,data = data_url(p.text)
                atts.append(Attachment(base64.b64decode(data), mime))
                segs.append(f'![](attachment:{atts[-1].id})')
            else: raise ValueError(f'unsupported user part: {p.type}')
        meta = getattr(u, 'meta', None)
        uid = (meta or {}).get('uid') or f'{name}\x1f{i}'
        kw = dict(id=hashlib.md5(uid.encode()).hexdigest()[:8])
        if meta: kw['meta'] = meta
        if segs and (mt := _env_pat.search(segs[-1])):
            segs[-1] = segs[-1][:mt.start()] + mt['task']
            if mt['sid']: kw['id'] = mt['sid']
        dlg.mk_message('\n\n'.join(segs), msg_type=sprompt, output=hist2fmt(replies, mx=mx), attachments=atts, **kw)
    return dlg

In [ ]:
rdlg = chat2dlg(dlg2chat(sdlg), 'seq round-trip')
test_eq(len(rdlg.messages), 1)
test_eq(rdlg.messages[0].content, 'Two steps.')
test_eq(dlg2chat(rdlg), sm)

A `Msg` can carry provenance `meta` -- a plain dynamic attribute, not a field of the class: `recs2chat` and `items2chat` stamp each canonical message with its source record's `created` time and `uid`, and any other producer may do the same. `chat2dlg` copies a user `Msg`'s meta onto the dialog message it becomes, and derives a stable 8-hex id from the `uid`, so a regenerated dialog keeps identical message ids and every message can cite its source record. An embedded solveit `sid` still wins the id when present:

In [ ]:
mm = mk_msg('When was this?')
mm.meta = dict(created='2026-07-09T00:26:40.000Z', uid='abc-123')
mdlg = chat2dlg([mm, Msg('assistant', [Text('July 9.')])], 'meta')
m0 = mdlg.messages[0]
test_eq(m0.meta, mm.meta)
test_eq(m0.id, hashlib.md5(b'abc-123').hexdigest()[:8])
m0.meta

{'created': '2026-07-09T00:26:40.000Z', 'uid': 'abc-123'}

Even without provenance meta, ids are deterministic -- derived from the dialog name and turn position -- so converting the same conversation twice yields identical ids rather than fresh random ones:

In [ ]:
d1,d2 = chat2dlg([mk_msg('a'), Msg('assistant', [Text('b')])], 'same'), chat2dlg([mk_msg('a'), Msg('assistant', [Text('b')])], 'same')
test_eq(d1.messages[0].id, d2.messages[0].id)

A transcript can begin mid-conversation -- a continuation session whose opening turns live in the predecessor file -- so a leading assistant run folds into a stub first prompt (empty content, replies attached) rather than raising:

In [ ]:
fdlg = chat2dlg([Msg('assistant', [Text('...continuing.')]), mk_msg('And then?'), Msg('assistant', [Text('Done.')])], 'cont')
test_eq(len(fdlg.messages), 2)
test_eq(fdlg.messages[0].content, '')
assert '...continuing.' in fdlg.messages[0].ai_res
fdlg.summary()

ba0dfd85:p:
> ...continuing.
753e7837:p:And then?
> Done.

In [ ]:
raw = [
    mk('prompt', 'hello there', output='Hello! How can I help?'),
    mk('note', 'a note between turns'),
    mk('code', output="aaa", content="'some source text'"),
    mk('prompt', 'add the numbers', output='They sum to 4.'),
    mk('note'), 
    mk('note'), 
    mk('code', output="333"),
    mk('prompt'),  # empty output on purpose
    mk('prompt', 'summarize the conversation so far', output='We greeted, then added some numbers.'),
    mk('note', 'below we insert an image'),
    mk('code', 'img=..some code to read image bytes'),
    mk('prompt', 'Please tell me what you see in the image', output='I can see...'),
    mk('note', 'lets add an image note'),
    mk('prompt', 'Please tell me what you see in the 2 images in the above msg',output='i see two puppies!'),
    mk('prompt', 'And one final question.')]

In [ ]:
aim_info = dict(supports_vision=True)

A realistic message mix — prompts with and without preceding notes and code, an empty AI response, media references — checks the turn structure:

In [ ]:
*hist,p,_ = dlg2hist(raw, aim_info)

Lets take a look at the history:

In [ ]:
for idx, turn in enumerate(hist):
    display(Markdown(f"**======turn: {idx}======**"))
    display(turn)

**======turn: 0======**

['<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="904dc672">hello there</prompt>']

**======turn: 1======**

'Hello! How can I help?'

**======turn: 2======**

...

'I can see...'

**======turn: 10======**

['<markdown id="3d395b19">lets add an image note</markdown>',
 '<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="0fd99282">Please tell me what you see in the 2 images in the above msg</prompt>']

**======turn: 11======**

'i see two puppies!'

In [ ]:
p

['<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="e376676d">And one final question.</prompt>']

In [ ]:
test_eq(len(hist),12)  # `raw` has 6 prompts which means 12 user+ai turns

In [ ]:
hist[6]

['<instructions>Respond to the request in the `prompt` below.</instructions>\n<prompt sid="1e3a33bc">summarize the conversation so far</prompt>']

In [ ]:
# the last prompt is marked `last`, but rendering is identical: the same envelope for every prompt
test_eq(p, [raw[-1].prompt_txt()])
test_eq(hist[6], [raw[8].prompt_txt()])
test(p[0], '<instructions>', operator.contains)


A raw message carrying a `rec_kind` tag in its meta is session bookkeeping some codec kept for reversibility (e.g. `llmsurgery.ant`'s system records), not conversation content, so history projections skip it:

In [ ]:
tr = [mk('note', 'context'), mk('prompt', 'q', output='r')]
h1 = dlg2hist(tr, aim_info)
trd = Dialog(name='trd'); trd.messages = list(tr)
test_eq(dlg2hist(trd, aim_info), h1)  # a Dialog and its message list produce the same history
tr.insert(1, Message('', msg_type=sraw, meta=dict(rec_kind='system', rec=dict(type='system'))))
test_eq(dlg2hist(tr, aim_info), h1)  # tagged raws never leak into history

A message with `skipped` set is hidden from the AI: it stays in the dialog (visible to any UI, saved to the file) but every history projection drops it. `pinned` doesn't affect projection — it marks messages a host's eviction policy must keep.

In [ ]:
sk = [mk('note', 'context'), mk('code', 'x = 1', output='1'), mk('prompt', 'q', output='r')]
full = dlg2hist(sk, aim_info)
sk[1].skipped = 1
h = dlg2hist(sk, aim_info)
test_eq(h, dlg2hist([sk[0], sk[2]], aim_info))   # projects as if the skipped message weren't there
assert h != full
sk[1].skipped = 0
test_eq(dlg2hist(sk, aim_info), full)            # unskip restores it

In [ ]:
# test each user input for expected length
test_eq(len(hist[0]),1) # prompt w/o preceding msgs
test_eq(len(hist[2]),3) # prompt w preceding note and code
test_eq(len(hist[4]),4) # prompt w preceding notes (2) and code
test_eq(len(hist[6]),1) # prompt w/o preceding msgs
test_eq(len(hist[8]),3)  # [note, code, prompt]
test_eq(len(hist[10]),2)  # [note, prompt]

In [ ]:
# Test each AI output for expected length
for i in (1,3,7,9,11): test(len(hist[i]),0,operator.gt)
test_eq(hist[5],'<output result="pending" reason="incomplete"/>')  # empty ai response

## Reply sub-dialogs

A prompt's AI reply is one markdown string: prose interleaved with tool-call details blocks. `reply2dlg` explodes that string into a `Dialog` of note and code messages, so a reply's insides can be edited with the ordinary message operations, and `dlg2reply` implodes the sub-dialog back into reply markdown. Parsing goes through `fmt2hist`, so the projection normalizes a reply exactly where the history path does: token-usage, think, and server-tool blocks are dropped, and a reply ending in a call gains a trailing `.` note.

In [ ]:
#| export
def _dotted_name(s): return bool(s) and all(p.isidentifier() for p in s.split('.'))
def reply2dlg(pmsg):
    "Explode `pmsg`'s AI reply into a `Dialog` of note and code messages"
    sub = Dialog(name=pmsg.id)
    sub.msg_cls = type(pmsg)
    msgs = fmt2hist(pmsg.ai_res)
    trs = {p.id:p for m in msgs if m.role=='tool' for p in m.content}
    for m in msgs:
        if m.role=='tool': continue
        for p in m.content:
            if isinstance(p, ToolUse):
                if not _dotted_name(p.name): raise ValueError(f"not a dotted Python identifier: {p.name}")
                if bad := first(k for k in p.arguments if not k.isidentifier()): raise ValueError(f"not a Python identifier: {bad}")
                args = ', '.join(f'{k}={v!r}' for k,v in p.arguments.items())
                sub.mk_message(f"{p.name}({args})", output=code_output(trs[p.id].text), meta=dict(tool_id=p.id))
            elif isinstance(p, Text) and p.text: sub.mk_message(p.text, msg_type=snote)
    return sub

Each tool call becomes a code message whose source is a literal Python call with `repr`'d argument values. Dotted names such as `tools.write_stdin` stay intact. The source alone is the record: `ast` recovers the exact call dict, and nothing else stores the tool name. The result rides in the output list and the call id in `meta`. The fixture builds a reply with `hist2fmt` from known `Msg`s, so the expected string is constructed rather than scraped from a captured chat.

In [ ]:
tu1 = ToolUse(id='x1', name='python', arguments={'code':'1+1'})
tr1 = ToolResult(id='x1', name='python', text='2')
rmsgs = [Msg(role='assistant', content=[Text('Check:'), tu1]), Msg(role='tool', content=[tr1]),
    Msg(role='assistant', content=[Text('It is 2.')])]
pm = Message('What is 1+1?', msg_type=sprompt, output=hist2fmt(rmsgs, mx=None))
Markdown(pm.ai_res)

Check:

```json {.tool}
{
  "id": "x1",
  "name": "python",
  "args": {
    "code": "1+1"
  },
  "result": "2"
}
```

It is 2.

In [ ]:
sub = reply2dlg(pm)
sub


**872c8b75**


<details markdown='1'>

- Check:
- python(code='1+1') ⇒ [{'output_type': 'execute_result', 'metadata': {}, 'data': …
- It is 2.

</details>

In [ ]:
test_eq([m.msg_type for m in sub.messages], [snote, scode, snote])
test_eq(sub.messages[1].content, "python(code='1+1')")
test_eq(sub.messages[1].output, code_output('2'))
test_eq(sub.messages[1].meta['tool_id'], 'x1')
sub.messages[1]

7cc4dbf5:c:python(code='1+1') ⇒ out(102)

A dotted tool name round-trips too: `reply2dlg` renders it as the Python call it was, so the exploded code message is directly runnable.

In [ ]:
dtc = ToolUse('d1', 'tools.write_stdin', {'session_id':7, 'chars':'x'})
dpm = Message('Continue.', msg_type=sprompt, output=hist2fmt(
    [Msg(role='assistant', content=[dtc]), mk_tool_res_msg([dtc], ['ok']), Msg(role='assistant', content=[Text('Done.')])], mx=None))
dsub = reply2dlg(dpm)
test_eq(dsub.messages[0].content, "tools.write_stdin(session_id=7, chars='x')")
dsub.messages[0].content

"tools.write_stdin(session_id=7, chars='x')"

`dlg2reply` renders each code message back into a details block through `mk_tr_details` with truncation disabled (the summary line is regenerated), passes note content through verbatim, and joins with blank lines: the same bytes `hist2fmt` produces. For a fmt2hist-clean reply the round trip is exact.

In [ ]:
#| export
def _parse_call(src):
    "`(name, arguments)` parsed from literal call source `src`, or None if it isn't one"
    try: call = ast.parse(src.strip(), mode='eval').body
    except SyntaxError: return None
    if not (isinstance(call, ast.Call) and not call.args and all(kw.arg for kw in call.keywords)): return None
    name = ast.unparse(call.func)
    if not _dotted_name(name): return None
    return name, {kw.arg: literal_eval(kw.value) for kw in call.keywords}

def _msg2tr(m):
    "Tool-result `Part` recovered from code message `m`, whose source is the literal call"
    if (p := _parse_call(m.content)) is None: raise ValueError(f"not a keyword-only tool call: {m.content!r}")
    tid = m.meta.setdefault('tool_id', 'call_'+rtoken_hex(8))
    return ToolResult(id=tid, name=p[0], arguments=p[1], text=render_outputs_ai(m.output or []))

def dlg2reply(sub):
    "Implode a sub-dialog of note and code messages back into reply markdown"
    parts = [mk_tr_details(_msg2tr(m), mx=None).strip() if m.msg_type==scode else m.content.strip() for m in sub.messages]
    return '\n\n'.join(p for p in parts if p)

In [ ]:
test_eq(dlg2reply(sub), pm.ai_res)
test_eq(dlg2reply(dsub), dpm.ai_res)

Editing one argument in a code message changes exactly that argument in the reparsed call, and the call id is untouched:

In [ ]:
sub.messages[1].content = "python(code='1+2')"
tu = first(p for m in fmt2hist(dlg2reply(sub)) for p in m.content if isinstance(p, ToolUse))
test_eq(tu.arguments, {'code':'1+2'})
test_eq(tu.id, 'x1')

In [ ]:
#| export
@patch(as_prop=True)
def tool_call(self:Message):
    "Parsed `dict(name=..., arguments=...)` when this code message's source is a literal tool call, else None"
    if self.msg_type != scode: return None
    if (p := _parse_call(self.content)) is None: return None
    return dict(name=p[0], arguments=p[1])

`tool_call` is the read-only view of a code message's call, for UIs and review loops; unlike the render path it has no side effects (no id generation) and returns None rather than raising for anything that isn't a literal call:

In [ ]:
test_eq(sub.messages[1].tool_call, dict(name='python', arguments={'code':'1+2'}))
test_eq(sub.messages[0].tool_call, None)  # a note
test_eq(Message('x = 1', msg_type=scode).tool_call, None)  # not a call expression

A parallel volley renders as adjacent details blocks, and explodes to sequential code messages in document order (the trailing `.` note is `fmt2hist`'s guard against empty assistant text). `dlg2chat` sees equal histories before and after an explode/implode cycle:

In [ ]:
tus = [ToolUse(id=f'x{i}', name='simple_add', arguments=dict(a=i, b=1)) for i in (2,3,4)]
tres = [ToolResult(id=f'x{i}', name='simple_add', text=str(i+1)) for i in (2,3,4)]
vmsgs = [Msg(role='assistant', content=[Text('Adding.'), *tus]), Msg(role='tool', content=tres)]
vdlg = Dialog(name='volley')
vpm = vdlg.mk_message('Add some numbers.', msg_type=sprompt, output=hist2fmt(vmsgs, mx=None))
vsub = reply2dlg(vpm)
test_eq([m.msg_type for m in vsub.messages], [snote, scode, scode, scode, snote])

In [ ]:
v2 = Dialog(name='volley2')
v2.mk_message('Add some numbers.', msg_type=sprompt, output=dlg2reply(vsub), id=vpm.id)
test_eq(dlg2chat(v2), dlg2chat(vdlg))

New messages inserted into the sub-dialog appear in the reply; a new call gets a generated id that doesn't collide:

In [ ]:
vsub.mk_message('One more:', msg_type=snote)
nc = vsub.mk_message('simple_add(a=5, b=1)', output=code_output('6'))
r3 = dlg2reply(vsub)
assert 'One more:' in r3
ids = [p.id for m in fmt2hist(r3) for p in m.content if isinstance(p, ToolUse)]
test_eq(len(ids), len(set(ids)))
assert nc.meta['tool_id']

A reply with no tool calls round-trips as a single note. Tool names that aren't dotted Python identifiers raise on explode, and a code message whose source isn't a keyword-only call raises on implode:

In [ ]:
sub1 = reply2dlg(Message('hi', msg_type=sprompt, output='Just prose.'))
test_eq([m.msg_type for m in sub1.messages], [snote])
test_eq(dlg2reply(sub1), 'Just prose.')

In [ ]:
badp = [Msg(role='assistant', content=[ToolUse(id='b1', name='foo-bar')]),
    Msg(role='tool', content=[ToolResult(id='b1', name='foo-bar', text='x')])]
with expect_fail(ValueError, 'identifier'): reply2dlg(Message('p', msg_type=sprompt, output=hist2fmt(badp, mx=None)))
with expect_fail(ValueError, 'tool call'): dlg2reply(Dialog(name='b', messages=[Message('1+1', msg_type=scode)]))

## export -

In [ ]:
#| hide
from nbdev import nbdev_export
nbdev_export()